# PET: positron annihilation and image reconstruction

*medical · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/medical/pet_mlem)

A tracer such as FDG emits positrons; each one annihilates with an electron into two 511 keV photons flying back to back. A batch of simulated decays is followed to the detector ring, every coincidence is binned into a sinogram by its line of response, and the image is recovered with MLEM, $\lambda \leftarrow \lambda\,A^{\mathsf T}(y / A\lambda) / A^{\mathsf T}1$, where the whole scanner is one system matrix $A$ and each iteration is two matrix–vector products.

**Vector form**

$$e^+ + e^- \to \gamma\gamma,\quad \mathbf p_1 = -\mathbf p_2,\ E_\gamma = m_e c^2 = 511\,\text{keV},\qquad \boldsymbol\lambda^{k+1} = \frac{\boldsymbol\lambda^{k}}{A^{\mathsf T}\mathbf 1}\odot A^{\mathsf T}\frac{\mathbf y}{A\boldsymbol\lambda^{k} + \mathbf r}$$

**Index (tensor) form**

$$a_{ei} = d_{ei} + \sigma_+\xi_{ei},\quad u_{epi} = \pm(\cos\varphi_e, \sin\varphi_e),\qquad \bar y_b = A_{bj}\lambda_j + r_b,\qquad \lambda_j^{k+1} = \frac{\lambda_j^k}{A_{bj}1_b}\,A_{bj}\frac{y_b}{\bar y_b^{\,k}}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $d_{ei}$ | `multinomial(activity) + jitter` | (E, 2) | decay positions drawn from the activity map (hot lesions decay more often) |
| $a_{ei}$ | `positron_range(decay, sigma, gen)` | (E, 2) | annihilation point: the positron travels about a millimetre first |
| $u_{epi}$ | `annihilation_photons(phi)` | (E, 2, 2) | two photons $p = 1, 2$ in opposite directions, each $511$ keV |
| $h_{epi}$ | `ring_hits(a, u, R)  # einsum('ei,epi->ep')` | (E, 2, 2) | where each photon reaches the ring: solve $/a + t u/ = R$ for $t > 0$ |
| $y_b$ | `bincount(bin[kept])` | (K·L,) | sinogram: detected coincidences per line of response $(\theta, s)$ |
| $A_{bj}$ | `system_matrix(n, dev)` | (K·L, n·n) | probability that a decay in pixel $j$ is counted in bin $b$ (times attenuation) |
| $A_{bj}\lambda_j + r_b$ | `einsum('bj,j->b', A, lam) + r` | (K·L,) | repeated $j$ ⇒ summed: forward projection plus randoms/scatter |
| $A_{bj}\,q_b$ | `einsum('bj,b->j', A, q)` | (n·n,) | repeated $b$ ⇒ summed: back-projection along every LOR |
| $\lambda_j^{k+1}$ | `mlem_step(lam, A, y, r, sens)` | (n·n,) | multiplicative update: stays positive and keeps the total counts |

The two photons leave in opposite directions, so the decay lies somewhere on the line joining the two detectors that fire together: a line of response. Collecting millions of them gives the sinogram (each point of the body traces a sine curve in $(\theta, s)$). MLEM treats the counts as Poisson with mean $\bar y = A\lambda + r$ and climbs the log-likelihood $\sum_b y_b \log \bar y_b - \bar y_b$: each iteration forward-projects the current image, compares with the data, and back-projects the ratio. It never goes negative, it conserves the total counts ($\sum_b (A\lambda)_b = \sum_b y_b$ without background), and its likelihood never decreases. Early iterations are smooth; later ones are sharper but amplify the Poisson noise, so the error first drops and then rises again (with the defaults the minimum is near 50 iterations; try 150 iterations, or fewer decays, to see the rise): clinical scanners stop early (or use OSEM subsets and a post-filter). FBP filters every projection with a ramp and back-projects in one step; it is fast but noisier and can go negative. Photons crossing the body are attenuated by $e^{-\int \mu\, dl}$ with $\mu \approx 0.096$ cm$^{-1}$ for water at 511 keV, and that factor depends only on the LOR, not on where the decay was, so it is folded into $A$. Simplifications: 2D only, Gaussian positron range (F-18 has a mean range of about 0.6 mm, Ga-68 about 3 mm, so it is hidden by the 6 mm pixels unless you raise it), no photon acollinearity, ideal crystals (each LOR end is placed at a random point of the crystal that fired) with no depth of interaction or dead time, and a known smooth background for randoms and scatter. All values are approximate and for education only, not for clinical use.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/medical/pet_mlem/main.py
"""Main loop: phantom → simulated coincidences → sinogram → MLEM / FBP reconstruction → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    gen = torch.Generator().manual_seed(0)
    n = params["grid"]
    K, L, _ = tensors.sinogram_bins(n)

    activity, mu = tensors.phantom(n, dev)  # (n, n) each
    A_geo = tensors.system_matrix(n, dev)  # (K·L, n·n) detection probabilities without attenuation
    if params["attenuation"] == "on":
        att = torch.exp(-tensors.path_lengths(A_geo, n) @ mu.flatten())  # (B,) survival exp(-∫μ dl) per LOR
    else:
        att = torch.ones(K * L, device=dev)
    A = att[:, None] * A_geo  # attenuation folded into the system model

    n_decays = params["kilo_decays"] * 1000
    events = solve.simulate_events(activity, n_decays, params["positron_range"] / 10, params["n_detectors"], gen)
    kept, trues = solve.detect(events, att, K * L, gen)
    bg = params["background"]
    r = bg / (1 - bg) * trues.sum() * tensors.background_shape(n, dev)  # (B,) expected randoms + scatter
    y = trues + torch.poisson(r.cpu(), generator=gen).to(dev)  # measured sinogram

    images, ll, totals = solve.mlem(A, y, r, params["n_iter"])
    recon_fbp = solve.fbp(A_geo, (y - r) / att, K, L)
    truth = activity.flatten() * n_decays / activity.sum()  # expected decays per pixel
    err = solve.relative_error(images, truth)
    err_fbp = solve.relative_error(recon_fbp, truth).item()

    print(f"{n_decays} decays → {int(kept.sum())} detected coincidences ({100 * kept.float().mean():.1f}%)")
    print(f"sinogram {K} angles × {L} distances, background {100 * (y.sum() - trues.sum()) / y.sum():.0f}% of counts")
    print(f"system matrix A {tuple(A.shape)}, image {n}×{n}")
    best = int(err.argmin())
    print(f"MLEM error: {err[1]:.3f} after 1 it, best {err[best]:.3f} at it {best}, {err[-1]:.3f} at it {len(err) - 1}")
    print(f"FBP error: {err_fbp:.3f}")
    rising = bool((ll.diff() >= -1e-6 * ll.abs()[1:]).all())  # float32 round-off allowed
    print(f"log-likelihood {ll[0]:.4g} → {ll[-1]:.4g}, never decreasing: {rising}")
    print(f"counts Σ Aλ = {totals[-1]:.0f} vs Σ (y - r) = {(y - r).sum():.0f}")
    return plot.animate(events, kept, y, activity, images, recon_fbp, err, err_fbp, n, out_dir / "pet_mlem.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/medical/pet_mlem/equation.py
r"""Physics → tensor translation.

1. Decay and annihilation (for a batch of events e, components i ∈ {x, y}, photons p ∈ {1, 2}):

    a_{ei} = d_{ei} + \sigma_+ \xi_{ei}                       positron range: small random step ξ ~ N(0, 1)
    u_{epi} = \pm (\cos\varphi_e, \sin\varphi_e)               two 511 keV photons, back to back
    h_{epi} = a_{ei} + t_{ep} u_{epi},   |h_{ep}| = R           where each photon hits the detector ring

   The ring hits h_{ep} define the line of response (LOR) with sinogram coordinates (θ, s):
   s = h_{e1,i} n_i(θ) is the signed distance of the LOR from the centre.

2. Reconstruction.  The expected counts in sinogram bin b are  \bar y_b = A_{bj} \lambda_j + r_b
   (repeated j summed: a matrix–vector product), where A_{bj} is the probability that a decay in pixel j
   is detected in bin b and r_b is the random/scatter background.  MLEM maximises the Poisson likelihood
   with the multiplicative update

    \lambda_j^{k+1} = \frac{\lambda_j^k}{A_{bj} 1_b} \; A_{bj} \frac{y_b}{A_{bj'} \lambda_{j'}^k + r_b}

   in which every A_{bj} … _b is summed over the bins b (a back-projection, einsum('bj,b->j')).
"""

import math

import torch

E_PHOTON = 511.0  # keV = electron rest energy m_e c²


def positron_range(decay: torch.Tensor, sigma: float, generator: torch.Generator) -> torch.Tensor:
    r"""a_{ei} = d_{ei} + σ ξ_{ei}: the positron travels a short random distance before annihilating.  (E, 2)"""
    xi = torch.randn(decay.shape, generator=generator).to(decay.device)
    return decay + sigma * xi


def annihilation_photons(phi: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    r"""Directions u_{epi} (E, 2, 2) and energies (E, 2) of the photon pair.

    The positron and electron are (almost) at rest, so momentum conservation sends the two photons in
    exactly opposite directions, and energy conservation gives each one m_e c² = 511 keV.
    """
    u = torch.stack([torch.cos(phi), torch.sin(phi)], dim=-1)  # (E, 2) photon 1
    directions = torch.stack([u, -u], dim=1)  # (E, 2 photons, 2 components)
    energies = torch.full(phi.shape + (2,), E_PHOTON, device=phi.device)
    return directions, energies


def ring_hits(a: torch.Tensor, u: torch.Tensor, R: float) -> torch.Tensor:
    r"""h_{epi} = a_{ei} + t_{ep} u_{epi} with |h| = R and t > 0.   a: (E, 2), u: (E, 2, 2) → (E, 2, 2).

    |a + t u|² = R²  ⇒  t = -a_i u_i + \sqrt{(a_i u_i)² - a_i a_i + R²}   (repeated i summed)
    """
    au = torch.einsum("ei,epi->ep", a, u)  # (E, 2)
    aa = torch.einsum("ei,ei->e", a, a)  # (E,)
    t = -au + torch.sqrt(au**2 - aa[:, None] + R**2)
    return a[:, None, :] + t[..., None] * u


def lor_coordinates(h1: torch.Tensor, h2: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    r"""Sinogram coordinates of the line through h1 and h2 (each (E, 2)): angle θ ∈ [0, π) of the normal
    n = (cos θ, sin θ), and signed distance s = h1_i n_i from the centre."""
    d = h2 - h1
    theta = torch.remainder(torch.atan2(d[:, 1], d[:, 0]) - math.pi / 2, math.pi)  # normal ⟂ line, folded
    n = torch.stack([torch.cos(theta), torch.sin(theta)], dim=-1)
    s = torch.einsum("ei,ei->e", h1, n)
    return theta, s


def forward_project(A: torch.Tensor, lam: torch.Tensor, r: torch.Tensor) -> torch.Tensor:
    r"""\bar y_b = A_{bj} λ_j + r_b   (repeated j summed).   A: (B, J), λ: (J,) → (B,)"""
    return torch.einsum("bj,j->b", A, lam) + r


def back_project(A: torch.Tensor, q: torch.Tensor) -> torch.Tensor:
    r"""A_{bj} q_b   (repeated b summed): smear every bin value back along its LOR.   → (J,)"""
    return torch.einsum("bj,b->j", A, q)


def mlem_step(lam: torch.Tensor, A: torch.Tensor, y: torch.Tensor, r: torch.Tensor, sens: torch.Tensor) -> torch.Tensor:
    r"""λ^{k+1}_j = λ^k_j / s_j · A_{bj} (y_b / ȳ_b),   with sensitivity s_j = A_{bj} 1_b."""
    ratio = y / forward_project(A, lam, r).clamp(min=1e-12)  # (B,) measured / expected
    return lam / sens.clamp(min=1e-12) * back_project(A, ratio)


def log_likelihood(y: torch.Tensor, y_bar: torch.Tensor) -> torch.Tensor:
    r"""Poisson log-likelihood (without the constant log y!):  L = y_b \log ȳ_b - ȳ_b, summed over b."""
    return (y * torch.log(y_bar.clamp(min=1e-12)) - y_bar).sum()


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/medical/pet_mlem/tensors.py
"""Build the concrete tensors: image grid, activity/attenuation phantom, sinogram bins and the system matrix A."""

import math

import torch

WIDTH = 30.0  # cm, side of the square image (field of view)
RING_RADIUS = 40.0  # cm, radius of the detector ring
MU_WATER = 0.096  # 1/cm, attenuation of 511 keV photons in water / soft tissue


def pixel_centres(n: int, device: torch.device) -> torch.Tensor:
    """Centres of the n×n pixels, shape (n, n, 2) with [iy, ix] = (x, y) in cm."""
    pixel = WIDTH / n
    c = -WIDTH / 2 + pixel * (torch.arange(n, device=device) + 0.5)
    yy, xx = torch.meshgrid(c, c, indexing="ij")
    return torch.stack([xx, yy], dim=-1)


def phantom(n: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Activity (relative uptake) and attenuation μ (1/cm), both (n, n): a body with two lungs and hot lesions."""
    p = pixel_centres(n, device)
    x, y = p[..., 0], p[..., 1]

    def ellipse(cx, cy, ax, ay):
        return ((x - cx) / ax) ** 2 + ((y - cy) / ay) ** 2 <= 1

    body = ellipse(0, 0, 13, 9.5)
    lungs = ellipse(-5.5, 1, 3.5, 5) | ellipse(5.5, 1, 3.5, 5)
    activity = 1.0 * body.float()
    activity[lungs] = 0.25
    activity[ellipse(0, -5.5, 1.8, 1.8)] = 0.0  # cold spine
    for cx, cy, rad, hot in [(-5.0, 2.5, 1.2, 8.0), (5.5, -0.5, 1.6, 5.0), (2.0, 5.0, 0.8, 10.0), (-1, -1.5, 2.0, 3.0)]:
        activity[ellipse(cx, cy, rad, rad)] = hot  # "hot" lesions (tumours take up more tracer)
    mu = MU_WATER * body.float()
    mu[lungs] = 0.03
    return activity, mu


def sinogram_bins(n: int) -> tuple[int, int, float]:
    """(n_theta, n_s, s_max): angles in [0, π) and distances in [-s_max, s_max] covering the whole image."""
    s_max = WIDTH / math.sqrt(2)
    return int(round(1.25 * n)), int(math.ceil(2 * s_max / (WIDTH / n))), s_max


def system_matrix(n: int, device: torch.device) -> torch.Tensor:
    """A_{bj}: probability that a decay in pixel j lands in sinogram bin b = (θ_k, s_l).  Shape (K·L, n·n).

    Each bin's line is sampled at many points; every sample adds its step length to the pixel it falls in,
    which approximates the length of the line inside each pixel.  Dividing by K·pixel²/Δs turns lengths into
    probabilities, so each pixel's column sums to ≈ 1 (every decay gives one LOR at a random angle).
    """
    K, L, s_max = sinogram_bins(n)
    pixel = WIDTH / n
    ds = 2 * s_max / L
    theta = (torch.arange(K, device=device) + 0.5) * math.pi / K  # (K,)
    s = -s_max + ds * (torch.arange(L, device=device) + 0.5)  # (L,)
    step = pixel / 4
    t = torch.arange(-s_max, s_max, step, device=device)  # (M,) position along the line
    normal = torch.stack([torch.cos(theta), torch.sin(theta)], -1)  # (K, 2)
    along = torch.stack([-torch.sin(theta), torch.cos(theta)], -1)  # (K, 2)
    # points p_{klm i} = s_l n_{ki} + t_m u_{ki}
    pts = s[None, :, None, None] * normal[:, None, None, :] + t[None, None, :, None] * along[:, None, None, :]
    ix = torch.floor((pts[..., 0] + WIDTH / 2) / pixel).long()
    iy = torch.floor((pts[..., 1] + WIDTH / 2) / pixel).long()
    inside = (ix >= 0) & (ix < n) & (iy >= 0) & (iy < n)
    rows = torch.arange(K * L, device=device).view(K, L, 1).expand_as(ix)
    A = torch.zeros(K * L, n * n, device=device)
    lengths = torch.full((int(inside.sum()),), step, device=device)
    A.index_put_((rows[inside], (iy * n + ix)[inside]), lengths, accumulate=True)  # repeated pixels add up
    return A * ds / (K * pixel**2)


def path_lengths(A: torch.Tensor, n: int) -> torch.Tensor:
    """Undo the probability scaling of A: geometric length (cm) of each LOR inside each pixel, (B, J)."""
    K, L, s_max = sinogram_bins(n)
    return A * K * (WIDTH / n) ** 2 / (2 * s_max / L)


def background_shape(n: int, device: torch.device) -> torch.Tensor:
    """Shape of randoms + scatter in the sinogram (sums to 1): half flat (randoms), half broad in s (scatter)."""
    K, L, s_max = sinogram_bins(n)
    s = -s_max + 2 * s_max / L * (torch.arange(L, device=device) + 0.5)
    profile = 0.5 / L + 0.5 * torch.softmax(-((s / 10.0) ** 2), dim=0)  # (L,)
    return (profile / K).repeat(K)  # (K·L,) bins ordered as b = k·L + l


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/medical/pet_mlem/solve.py
"""Monte Carlo of decays → coincidences → sinogram, then MLEM (and FBP for comparison) reconstruction."""

import math

import torch

from problems._common.progress import steps

from . import tensors
from .equation import (
    annihilation_photons,
    back_project,
    forward_project,
    log_likelihood,
    lor_coordinates,
    mlem_step,
    positron_range,
    ring_hits,
)


def simulate_events(
    activity: torch.Tensor, n_decays: int, sigma: float, n_detectors: int, generator: torch.Generator
) -> dict[str, torch.Tensor]:
    """Follow every decay to a detected coincidence.  Returns per-event tensors (E = n_decays):

    decay, annihilation (E, 2) · hits on the ring (E, 2, 2) · crystal centres (E, 2, 2) · sinogram bin (E,)
    · in_sinogram (E,) bool · energies (E, 2) · directions (E, 2, 2)
    """
    dev = activity.device
    n = activity.shape[0]
    pixel = tensors.WIDTH / n
    # 1. where the tracer decays: pick pixels in proportion to activity, then a random point inside the pixel
    j = torch.multinomial(activity.flatten().cpu(), n_decays, replacement=True, generator=generator).to(dev)
    jitter = (torch.rand((n_decays, 2), generator=generator).to(dev) - 0.5) * pixel
    decay = tensors.pixel_centres(n, dev).view(-1, 2)[j] + jitter
    # 2. positron range, 3. back-to-back photons at a random angle, 4. where they hit the ring
    annihilation = positron_range(decay, sigma, generator)
    phi = 2 * math.pi * torch.rand(n_decays, generator=generator).to(dev)
    directions, energies = annihilation_photons(phi)
    hits = ring_hits(annihilation, directions, tensors.RING_RADIUS)  # (E, 2, 2)
    # 5. the detector ring is made of n_detectors crystals: we only know WHICH crystal fired, so the LOR end is
    #    placed at a random point of that crystal (snapping to the crystal centre would leave sinogram bins
    #    unevenly filled → ring artefacts; real scanners remove those with a measured normalisation)
    width = 2 * math.pi / n_detectors
    crystal = torch.floor(torch.remainder(torch.atan2(hits[..., 1], hits[..., 0]), 2 * math.pi) / width)
    angle = (crystal + torch.rand(crystal.shape, generator=generator).to(dev)) * width
    crystals = tensors.RING_RADIUS * torch.stack([torch.cos(angle), torch.sin(angle)], -1)  # (E, 2, 2)
    # 6. bin the LOR into the sinogram (θ_k, s_l)
    theta, s = lor_coordinates(crystals[:, 0], crystals[:, 1])
    K, L, s_max = tensors.sinogram_bins(n)
    k = torch.clamp(torch.floor(theta / (math.pi / K)).long(), max=K - 1)
    ell = torch.floor((s + s_max) / (2 * s_max / L)).long()
    in_sino = (ell >= 0) & (ell < L)
    return {
        "decay": decay,
        "annihilation": annihilation,
        "directions": directions,
        "energies": energies,
        "hits": hits,
        "crystals": crystals,
        "bin": torch.where(in_sino, k * L + ell, 0),
        "in_sinogram": in_sino,
    }


def detect(
    events: dict, attenuation: torch.Tensor, n_bins: int, generator: torch.Generator
) -> tuple[torch.Tensor, torch.Tensor]:
    """Keep each coincidence with probability exp(-∫μ dl) along its LOR; return (kept mask (E,), sinogram (B,))."""
    survive = torch.rand(events["bin"].shape, generator=generator).to(attenuation.device) < attenuation[events["bin"]]
    kept = events["in_sinogram"] & survive
    return kept, torch.bincount(events["bin"][kept], minlength=n_bins).float()


def mlem(
    A: torch.Tensor, y: torch.Tensor, r: torch.Tensor, n_iter: int
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Run MLEM from a flat image.  Returns (images (n_iter+1, J), log-likelihood (n_iter+1,),
    total expected counts Σ_b A λ (n_iter+1,))."""
    sens = back_project(A, torch.ones_like(y))  # (J,) s_j = A_{bj} 1_b
    lam = torch.full_like(sens, ((y - r).sum().clamp(min=1) / sens.sum()).item())  # flat start, right total
    images, ll, totals = [lam], [], []
    for _ in steps(n_iter, "MLEM iterations"):
        lam = mlem_step(lam, A, y, r, sens)
        images.append(lam)
    for lam in images:
        y_bar = forward_project(A, lam, r)
        ll.append(log_likelihood(y, y_bar))
        totals.append((y_bar - r).sum())
    return torch.stack(images), torch.stack(ll), torch.stack(totals)


def ramp_filter(sino: torch.Tensor) -> torch.Tensor:
    """Filter every projection (rows of a (K, L) sinogram) with a ramp |ν| × Hann window, via the FFT."""
    L = sino.shape[1]
    nu = torch.fft.fftfreq(2 * L, device=sino.device)  # zero-padded to avoid wrap-around
    H = nu.abs() * 0.5 * (1 + torch.cos(2 * math.pi * nu))  # Hann apodisation tames the noise
    return torch.fft.ifft(torch.fft.fft(sino, n=2 * L, dim=1) * H, dim=1).real[:, :L]


def fbp(A_geo: torch.Tensor, y_corr: torch.Tensor, K: int, L: int) -> torch.Tensor:
    """Filtered back-projection of corrected data y_corr (B,), scaled so its forward projection has the same counts."""
    image = back_project(A_geo, ramp_filter(y_corr.view(K, L)).flatten())
    return image * y_corr.sum() / forward_project(A_geo, image, torch.zeros_like(y_corr)).sum()


def relative_error(images: torch.Tensor, truth: torch.Tensor) -> torch.Tensor:
    """‖λ - λ_true‖ / ‖λ_true‖ for each image in a batch (…, J)."""
    return (images - truth).norm(dim=-1) / truth.norm()


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/medical/pet_mlem/plot.py
"""Animate photon pairs reaching the ring, the sinogram filling up, then MLEM sharpening iteration by iteration."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.collections import LineCollection

from problems._common.plotting import frame_indices, save_gif

from . import tensors

FILL_STEPS = 40  # animation steps spent collecting data before the reconstruction starts
N_SHOWN = 160  # coincidences drawn as lines on the ring


def animate(
    events: dict,
    kept: torch.Tensor,
    y: torch.Tensor,
    activity: torch.Tensor,
    images: torch.Tensor,
    recon_fbp: torch.Tensor,
    err: torch.Tensor,
    err_fbp: float,
    n: int,
    path: Path,
) -> Path:
    K, L, s_max = tensors.sinogram_bins(n)
    n_iter = images.shape[0] - 1
    half, R = tensors.WIDTH / 2, tensors.RING_RADIUS
    # events shown on the ring: the first N_SHOWN detected coincidences
    idx = torch.nonzero(kept).flatten()
    shown = idx[:N_SHOWN]
    a = events["annihilation"][shown].cpu().numpy()  # (S, 2)
    hits = events["crystals"][shown].cpu().numpy()  # (S, 2, 2)
    # cumulative sinograms while the data come in (detected events in the order they were simulated)
    bins = events["bin"][idx]
    background = (y - torch.bincount(bins, minlength=K * L).float()).clamp(min=0)
    cuts = [round(len(bins) * (i + 1) / FILL_STEPS) for i in range(FILL_STEPS)]
    partial = [
        (torch.bincount(bins[:c], minlength=K * L).float() + background * c / len(bins)).view(K, L).T.cpu().numpy()
        for c in cuts
    ]
    imgs = images.view(-1, n, n).cpu().numpy()
    fbp_img = recon_fbp.view(n, n).cpu().numpy()
    truth = (activity / activity.sum() * images[0].sum()).cpu().numpy()  # true image on the same count scale
    vmax = float(truth.max()) * 1.1
    cmap = plt.get_cmap("inferno", 16)
    img_kw = {"origin": "lower", "cmap": cmap, "vmin": 0, "vmax": vmax, "extent": [-half, half, -half, half]}

    fig, axes = plt.subplots(2, 3, figsize=(9, 5.8))
    (ax_ring, ax_sino, ax_true), (ax_fbp, ax_mlem, ax_err) = axes
    for ax in (ax_ring, ax_true, ax_fbp, ax_mlem):
        ax.set_aspect("equal")
        ax.grid(False)
        ax.set_xticks([])
        ax.set_yticks([])

    # --- ring with decays and lines of response ---------------------------------------------------------
    body = np.ma.masked_equal(activity.cpu().numpy(), 0)  # transparent outside the body
    ax_ring.imshow(body, alpha=0.35, **{**img_kw, "vmax": float(activity.max())})
    ring = np.linspace(0, 2 * np.pi, 200)
    ax_ring.plot(R * np.cos(ring), R * np.sin(ring), color="#374151", lw=4)
    ax_ring.set_xlim(-R * 1.08, R * 1.08)
    ax_ring.set_ylim(-R * 1.08, R * 1.08)
    old = LineCollection([], colors="#2563eb", linewidths=0.5, alpha=0.35)
    new = LineCollection([], colors="#dc2626", linewidths=1.4)
    ax_ring.add_collection(old)
    ax_ring.add_collection(new)
    flash = ax_ring.plot([], [], "o", color="#dc2626", ms=4)[0]
    origin = ax_ring.plot([], [], "*", color="#f59e0b", ms=8)[0]
    ring_title = ax_ring.set_title("coincidences: 0")

    # --- sinogram ---------------------------------------------------------------------------------------
    sino = ax_sino.imshow(
        partial[0], origin="lower", aspect="auto", cmap=plt.get_cmap("viridis", 16), vmin=0,
        vmax=float(partial[-1].max()), extent=[0, 180, -s_max, s_max],
    )  # fmt: skip
    ax_sino.set_xlabel("θ (°)")
    ax_sino.set_ylabel("s (cm)")
    ax_sino.grid(False)
    sino_title = ax_sino.set_title("sinogram (LOR counts)")

    ax_true.imshow(truth, **img_kw)
    ax_true.set_title("true activity")
    fbp_im = ax_fbp.imshow(np.zeros_like(fbp_img), **img_kw)
    fbp_title = ax_fbp.set_title("FBP (waiting for data)")
    mlem_im = ax_mlem.imshow(imgs[0], **img_kw)
    mlem_title = ax_mlem.set_title("MLEM, iteration 0")

    # --- convergence --------------------------------------------------------------------------------------
    e = err.cpu().numpy()
    ax_err.set_xlim(0, max(n_iter, 1))
    ax_err.set_ylim(0, max(float(e.max()), err_fbp) * 1.1)
    ax_err.axhline(err_fbp, ls="--", color="#16a34a", lw=1.2, label="FBP")
    err_line = ax_err.plot([], [], color="#dc2626", lw=1.8, label="MLEM")[0]
    ax_err.set_xlabel("iteration")
    ax_err.set_title("error vs. true activity")
    ax_err.legend(fontsize=8)

    def update(step: int):
        if step < FILL_STEPS:
            m = round(N_SHOWN * (step + 1) / FILL_STEPS)
            m_old = round(N_SHOWN * step / FILL_STEPS)
            old.set_segments(list(hits[:m_old]))
            # the photon pairs that just arrived: two back-to-back paths from the annihilation point to the ring
            new.set_segments([np.stack([a[e], h]) for e in range(m_old, m) for h in hits[e]])
            flash.set_data(hits[m_old:m, :, 0].ravel(), hits[m_old:m, :, 1].ravel())
            origin.set_data(a[m_old:m, 0], a[m_old:m, 1])
            sino.set_data(partial[step])
            ring_title.set_text(f"coincidences: {cuts[step]:,}")
            return
        # all data are in: final ring picture and sinogram, FBP appears, MLEM iterates
        k = step - FILL_STEPS + 1
        old.set_segments(list(hits))
        new.set_segments([])
        flash.set_data([], [])
        origin.set_data([], [])
        sino.set_data(partial[-1])
        ring_title.set_text(f"coincidences: {cuts[-1]:,}")
        sino_title.set_text("sinogram (complete)")
        fbp_im.set_data(fbp_img)
        fbp_title.set_text("FBP (ramp × Hann)")
        mlem_im.set_data(imgs[k])
        mlem_title.set_text(f"MLEM, iteration {k}")
        err_line.set_data(np.arange(k + 1), e[: k + 1])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(FILL_STEPS + n_iter, 100), path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'kilo_decays': 1500,
    'n_iter': 40,
    'positron_range': 1.0,
    'attenuation': 'on',
    'background': 0.2,
    'grid': 48,
    'n_detectors': 256
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.medical.pet_mlem")]:
    del sys.modules[name]
main = importlib.import_module("problems.medical.pet_mlem.main")
problem_dir = Path("problems/medical/pet_mlem")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
